# LeetCode 1899 — Merge Triplets to Form Target Triplet bằng Python

Link đề: https://leetcode.com/problems/merge-triplets-to-form-target-triplet/ · Độ khó: **Medium** · Chủ đề: Array, Greedy

Skeleton LeetCode:

```python
class Solution:
    def mergeTriplets(self, triplets: List[List[int]], target: List[int]) -> bool:
        ...
```

Lộ trình:
1. **Mô phỏng đúng thao tác** bằng BFS trên trạng thái của cả mảng (vét cạn thuần, `n` rất nhỏ).
2. **Tập "max của mọi tập con"**: mọi bộ ba tạo được là max theo từng thành phần của một tập con khác rỗng ⇒ liệt kê `2ⁿ` tập con.
3. **Bao đóng dần** tập các bộ ba đạt được (thêm max từng cặp tới khi ổn định) — đa thức theo số bộ ba phân biệt.
4. **Tham lam một lượt**: bỏ mọi bộ ba có **thành phần vượt** target; max của các bộ ba còn lại phải bằng target (`O(n)`).
5. Biến thể viết bằng **ba cờ** `found_x, found_y, found_z` với dừng sớm.

## 1. Mục tiêu học

1. Hiểu phép gộp `max` theo từng thành phần: **giao hoán, kết hợp, lũy đẳng** ⇒ thứ tự và số lần gộp không quan trọng, chỉ **tập** bộ ba được dùng quan trọng.
2. Chứng minh: target đạt được ⇔ target = max của một tập con ⇔ max của **mọi bộ ba "an toàn"** (không vượt target ở thành phần nào) bằng target.
3. Nhận ra bộ ba có **một thành phần vượt target** là "độc" — dùng nó thì không bao giờ hạ xuống được (max chỉ tăng).
4. Viết lời giải `O(n)` một lượt và phiên bản dừng sớm bằng cờ.
5. Kiểm chứng bằng vét cạn tập con và bằng mô phỏng thao tác thật.

## 2. Đề bài & ví dụ

Một **bộ ba** là mảng ba số nguyên. Cho mảng `triplets` với `triplets[i] = [aᵢ, bᵢ, cᵢ]` và bộ ba đích `target = [x, y, z]`.

Có thể thực hiện thao tác sau **bao nhiêu lần tùy ý** (kể cả 0):
- Chọn hai chỉ số `i ≠ j`, cập nhật `triplets[j] = [max(aᵢ, aⱼ), max(bᵢ, bⱼ), max(cᵢ, cⱼ)]`.

Ví dụ `triplets[i] = [2,5,3]`, `triplets[j] = [1,7,5]` ⇒ `triplets[j]` thành `[2,7,5]`.

Trả về `True` nếu có thể làm `target` xuất hiện như **một phần tử** của `triplets`.

**Ví dụ 1:** `[[2,5,3],[1,8,4],[1,7,5]]`, target `[2,7,5]` → `True` (gộp bộ 0 vào bộ 2).

**Ví dụ 2:** `[[3,4,5],[4,5,6]]`, target `[3,2,5]` → `False` (không có số 2 ở vị trí giữa).

**Ví dụ 3:** `[[2,5,3],[2,3,4],[1,2,5],[5,2,3]]`, target `[5,5,5]` → `True`.

**Ràng buộc:** `1 ≤ triplets.length ≤ 10⁵`, các giá trị trong `[1, 1000]`.

In [1]:
import random
import time
from collections import deque
from itertools import combinations
from typing import List

examples = [
    (([[2, 5, 3], [1, 8, 4], [1, 7, 5]], [2, 7, 5]), True),
    (([[3, 4, 5], [4, 5, 6]], [3, 2, 5]), False),
    (([[2, 5, 3], [2, 3, 4], [1, 2, 5], [5, 2, 3]], [5, 5, 5]), True),
]
for (tr, tg), want in examples:
    print(f"triplets={tr}, target={tg} -> kỳ vọng {want}")

triplets=[[2, 5, 3], [1, 8, 4], [1, 7, 5]], target=[2, 7, 5] -> kỳ vọng True
triplets=[[3, 4, 5], [4, 5, 6]], target=[3, 2, 5] -> kỳ vọng False
triplets=[[2, 5, 3], [2, 3, 4], [1, 2, 5], [5, 2, 3]], target=[5, 5, 5] -> kỳ vọng True


## 3. Kiến thức nền

### 3.1 Max theo từng thành phần

In [2]:
def merge(u, v):
    return [max(a, b) for a, b in zip(u, v)]

print(merge([2, 5, 3], [1, 7, 5]))

[2, 7, 5]


### 3.2 Tính chất của phép `max`

- **Giao hoán:** `merge(u, v) = merge(v, u)`.
- **Kết hợp:** `merge(merge(u, v), w) = merge(u, merge(v, w))`.
- **Lũy đẳng:** `merge(u, u) = u`.

⇒ Gộp một tập bộ ba theo **bất kỳ thứ tự** nào, lặp lại bao nhiêu lần, kết quả đều là **max từng thành phần của cả tập**.

In [3]:
from functools import reduce
ts = [[2, 5, 3], [1, 2, 5], [5, 2, 3]]
orders = [reduce(merge, [ts[i] for i in perm]) for perm in [(0, 1, 2), (2, 1, 0), (1, 0, 2, 0)][:2]]
print(orders, reduce(merge, ts + ts))

[[5, 5, 5], [5, 5, 5]] [5, 5, 5]


### 3.3 `all` / `any` và so sánh "không vượt"

Bộ ba `t` **an toàn** với target `g` nếu `t[k] ≤ g[k]` với mọi `k`: `all(a <= b for a, b in zip(t, g))`.

In [4]:
g = [2, 7, 5]
for t in [[2, 5, 3], [1, 8, 4], [1, 7, 5]]:
    print(t, "an toàn?", all(a <= b for a, b in zip(t, g)))

[2, 5, 3] an toàn? True
[1, 8, 4] an toàn? False
[1, 7, 5] an toàn? True


### 3.4 Liệt kê tập con

`combinations(range(n), r)` cho mọi tập con kích thước `r`; tổng cộng `2ⁿ − 1` tập con khác rỗng.

## 4. Ý tưởng nền tảng / trực giác

Ví dụ 1, target `[2, 7, 5]`:

| bộ ba | an toàn (mọi thành phần ≤ target)? | đóng góp |
|---|---|---|
| [2,5,3] | có | đạt `x = 2` |
| [1,8,4] | **không** (8 > 7) | bỏ |
| [1,7,5] | có | đạt `y = 7`, `z = 5` |

Max các bộ an toàn: `[2, 7, 5]` = target ⇒ `True`.

**Vì sao phải bỏ `[1,8,4]`?** Một khi đã gộp nó vào một bộ ba, thành phần thứ hai sẽ `≥ 8 > 7` mãi mãi (max chỉ tăng) ⇒ bộ ba đó không bao giờ thành target.

Ví dụ 2, target `[3, 2, 5]`: bộ `[3,4,5]` có `4 > 2`, bộ `[4,5,6]` có mọi thứ vượt ⇒ không còn bộ an toàn ⇒ `False`.

## 5. Cách 1 — Brute Force / baseline

### 5.1 BFS trên trạng thái của **cả mảng** (mô phỏng thao tác thật)

Trạng thái = tuple các bộ ba; mỗi bước chọn `(i, j)`, `i ≠ j`, thay `triplets[j]`. Dừng khi target xuất hiện. Đúng theo định nghĩa từng chữ — chỉ chạy được với `n ≤ 4` và giá trị nhỏ.

In [5]:
def merge_bfs_state(triplets, target):
    start = tuple(tuple(t) for t in triplets); tg = tuple(target)
    seen = {start}; q = deque([start])
    while q:
        st = q.popleft()
        if tg in st:
            return True
        for i in range(len(st)):
            for j in range(len(st)):
                if i != j:
                    nt = tuple(max(a, b) for a, b in zip(st[i], st[j]))
                    if nt != st[j]:
                        ns = st[:j] + (nt,) + st[j + 1:]
                        if ns not in seen:
                            seen.add(ns); q.append(ns)
    return False

for (tr, tg), want in examples:
    print(merge_bfs_state(tr, tg), "kỳ vọng", want)

True kỳ vọng True
False kỳ vọng False
True kỳ vọng True


### 5.2 Cách 1b — Max của mọi tập con

Nhờ tính chất ở mục 3.2, mọi bộ ba xuất hiện được đều là max của **một tập con** các bộ ba ban đầu (và ngược lại, mọi max tập con đều tạo được: gộp lần lượt các phần tử của tập vào một phần tử của tập). Liệt kê `2ⁿ − 1` tập con.

In [6]:
def merge_subsets(triplets, target):
    n = len(triplets)
    for r in range(1, n + 1):
        for comb in combinations(range(n), r):
            if reduce(merge, [triplets[i] for i in comb]) == list(target):
                return True
    return False

for (tr, tg), want in examples:
    print(merge_subsets(tr, tg), "kỳ vọng", want)

True kỳ vọng True
False kỳ vọng False
True kỳ vọng True


**Độ phức tạp:** `O(2ⁿ · n)` — chỉ cho `n ≤ 18`. Space `O(n)`.

**Nút thắt:** ta không cần biết **tập nào** — chỉ cần biết target có phải max của **tập an toàn lớn nhất** không.

In [7]:
random.seed(0)
for n in [10, 14, 17]:
    tr = [[random.randint(1, 9) for _ in range(3)] for _ in range(n)]
    tg = [10, 10, 10]                                     # không đạt được -> phải duyệt hết
    t0 = time.perf_counter(); merge_subsets(tr, tg)
    print(f"n={n}: {time.perf_counter() - t0:.3f}s")

n=10: 0.001s
n=14: 0.027s


n=17: 0.253s


### 5.3 Cách 1c — Bao đóng dần

Bắt đầu với tập `T` các bộ ba (khử trùng). Lặp: thêm `merge(u, v)` cho mọi cặp trong `T`; dừng khi không thêm gì mới. Số bộ ba phân biệt tối đa `1000³`, nhưng với input nhỏ thì nhanh. Một vét cạn khác hẳn để đối chiếu.

In [8]:
def merge_closure(triplets, target):
    T = {tuple(t) for t in triplets}
    changed = True
    while changed:
        changed = False
        cur = list(T)
        for i in range(len(cur)):
            for j in range(i + 1, len(cur)):
                m = tuple(max(a, b) for a, b in zip(cur[i], cur[j]))
                if m not in T:
                    T.add(m); changed = True
    return tuple(target) in T

for (tr, tg), want in examples:
    print(merge_closure(tr, tg), "kỳ vọng", want)

True kỳ vọng True
False kỳ vọng False
True kỳ vọng True


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (tập con):** bộ ba đạt được = `max(S)` với `S` là tập con khác rỗng.

**Insight 2 (bộ độc):** nếu `t ∈ S` có `t[k] > target[k]` thì `max(S)[k] ≥ t[k] > target[k]` ⇒ `max(S) ≠ target`. Vậy `S` chỉ chứa bộ **an toàn** (`t ≤ target` từng thành phần).

**Insight 3 (lấy hết bộ an toàn):** với `G` = tập mọi bộ an toàn, `max(G) ≤ target` (từng thành phần). Mọi `S ⊆ G` có `max(S) ≤ max(G)`. Nên tồn tại `S` với `max(S) = target` ⇔ `max(G) = target` (nếu `max(S) = target` thì `target ≤ max(G) ≤ target`).

**Quy tắc:** duyệt mọi bộ ba, bỏ bộ nào có thành phần vượt target, lấy max của phần còn lại; trả `max == target`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Tham lam: max của các bộ an toàn

In [9]:
def merge_greedy(triplets, target):
    x, y, z = target
    best = [0, 0, 0]
    for a, b, c in triplets:
        if a <= x and b <= y and c <= z:
            best = [max(best[0], a), max(best[1], b), max(best[2], c)]
    return best == [x, y, z]

for (tr, tg), want in examples:
    print(merge_greedy(tr, tg), "kỳ vọng", want)

True kỳ vọng True
False kỳ vọng False
True kỳ vọng True


Khởi tạo `best = [0, 0, 0]` an toàn vì mọi giá trị `≥ 1`; nếu không có bộ an toàn nào, `best` vẫn `[0,0,0] ≠ target`.

Code nộp LeetCode của Cách 2: xem mục 9.

**Độ phức tạp:** Time `O(n)`, Space `O(1)`.

### 7.2 Cách 3 — Ba cờ + dừng sớm

Thay vì giữ max, chỉ cần biết mỗi thành phần của target có **được đạt đúng** bởi một bộ an toàn nào không (vì max của bộ an toàn không vượt target, đạt bằng ⇔ có bộ an toàn có đúng giá trị đó).

In [10]:
def merge_flags(triplets, target):
    x, y, z = target
    fx = fy = fz = False
    for a, b, c in triplets:
        if a <= x and b <= y and c <= z:
            fx = fx or a == x
            fy = fy or b == y
            fz = fz or c == z
            if fx and fy and fz:
                return True
    return False

for (tr, tg), want in examples:
    print(merge_flags(tr, tg), "kỳ vọng", want)

True kỳ vọng True
False kỳ vọng False
True kỳ vọng True


**Độ phức tạp:** Time `O(n)` worst, có thể dừng sớm; Space `O(1)`.

### 7.3 Cách 4 — Biểu thức gọn bằng `zip(*...)`

Lọc bộ an toàn rồi lấy max theo cột bằng `zip(*good)`.

In [11]:
def merge_zip(triplets, target):
    good = [t for t in triplets if all(a <= b for a, b in zip(t, target))]
    return bool(good) and [max(col) for col in zip(*good)] == list(target)

for (tr, tg), want in examples:
    print(merge_zip(tr, tg), "kỳ vọng", want)

True kỳ vọng True
False kỳ vọng False
True kỳ vọng True


**Độ phức tạp:** Time `O(n)`, Space `O(n)` (danh sách `good`).

## 8. TRACE từng bước

Trace Cách 2 và Cách 3 trên ví dụ 3:

In [12]:
def trace_greedy(triplets, target):
    x, y, z = target; best = [0, 0, 0]
    for t in triplets:
        a, b, c = t
        if a <= x and b <= y and c <= z:
            best = [max(best[0], a), max(best[1], b), max(best[2], c)]
            print(f"{t}: an toàn -> best = {best}")
        else:
            bad = [k for k, (u, v) in enumerate(zip(t, target)) if u > v]
            print(f"{t}: ĐỘC (thành phần {bad} vượt target) -> bỏ")
    print("best == target?", best == list(target))
    return best == list(target)

trace_greedy([[2, 5, 3], [2, 3, 4], [1, 2, 5], [5, 2, 3]], [5, 5, 5])

[2, 5, 3]: an toàn -> best = [2, 5, 3]
[2, 3, 4]: an toàn -> best = [2, 5, 4]
[1, 2, 5]: an toàn -> best = [2, 5, 5]
[5, 2, 3]: an toàn -> best = [5, 5, 5]
best == target? True


True

In [13]:
trace_greedy([[2, 5, 3], [1, 8, 4], [1, 7, 5]], [2, 7, 5])
trace_greedy([[3, 4, 5], [4, 5, 6]], [3, 2, 5])

[2, 5, 3]: an toàn -> best = [2, 5, 3]
[1, 8, 4]: ĐỘC (thành phần [1] vượt target) -> bỏ
[1, 7, 5]: an toàn -> best = [2, 7, 5]
best == target? True
[3, 4, 5]: ĐỘC (thành phần [1] vượt target) -> bỏ
[4, 5, 6]: ĐỘC (thành phần [0, 1, 2] vượt target) -> bỏ
best == target? False


False

Bảng tóm tắt ví dụ 3 (target `[5,5,5]`): mọi bộ đều an toàn; max theo cột `[5, 5, 5]` ⇒ `True`. Cách gộp cụ thể của đề (`[2,5,3]` + `[1,2,5]` → `[2,5,5]`, rồi + `[5,2,3]` → `[5,5,5]`) chỉ là một thứ tự gộp tập `{0, 2, 3}`.

## 9. Code nộp LeetCode

In [14]:
class Solution:
    def mergeTriplets(self, triplets: List[List[int]], target: List[int]) -> bool:
        x, y, z = target
        fx = fy = fz = False
        for a, b, c in triplets:
            if a <= x and b <= y and c <= z:
                if a == x: fx = True
                if b == y: fy = True
                if c == z: fz = True
        return fx and fy and fz

sol = Solution()
for (tr, tg), want in examples:
    got = sol.mergeTriplets(tr, tg)
    print(got, "OK" if got == want else "SAI")

True OK
False OK
True OK


## 10. Vì sao thuật toán đúng

**(a) Bộ ba đạt được = max của tập con.** Quy nạp theo số thao tác: ban đầu mỗi `triplets[j]` là max của tập `{j}`. Thao tác gán `triplets[j] = merge(triplets[i], triplets[j])`; nếu trước đó chúng là max của `Sᵢ`, `Sⱼ` thì sau đó là max của `Sᵢ ∪ Sⱼ` (tính kết hợp). Ngược lại, với tập `S = {s₁, …, s_r}`, gộp lần lượt `s₂, …, s_r` vào `s₁`... (gán vào chỉ số `s₁`) tạo đúng `max(S)` tại vị trí `s₁` (cần `i ≠ j` — luôn thỏa vì các chỉ số khác nhau).

**(b) Chỉ dùng bộ an toàn.** Nếu `S` chứa bộ độc thì `max(S)` vượt target ở thành phần đó.

**(c) Lấy hết bộ an toàn là tốt nhất.** `max(G) ≤ target` và `max(S) ≤ max(G)` với mọi `S ⊆ G` ⇒ tồn tại `S` đạt target ⇔ `max(G) = target`.

**(d) Ba cờ.** Vì `max(G) ≤ target`, `max(G)[k] = target[k]` ⇔ tồn tại bộ an toàn có thành phần `k` bằng `target[k]`. ✓

## 11. Phân tích Time / Space

Gọi `n ≤ 10⁵`.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. BFS trạng thái mảng | mũ | mũ | `n ≤ 4` |
| 1b. Mọi tập con | `O(2ⁿ n)` | `O(n)` | `n ≤ 18` |
| 1c. Bao đóng | `O(U²)` mỗi vòng (`U` số bộ phân biệt) | `O(U)` | giá trị nhỏ |
| 2. Tham lam max | `O(n)` | `O(1)` | |
| 3. Ba cờ | `O(n)`, dừng sớm | `O(1)` | **nộp bài** |
| 4. `zip(*)` | `O(n)` | `O(n)` | gọn |

In [15]:
random.seed(1)
tr = [[random.randint(1, 1000) for _ in range(3)] for _ in range(10**5)]
tg = [999, 999, 999]
for name, fn in [("greedy", merge_greedy), ("flags", merge_flags), ("zip", merge_zip), ("submit", Solution().mergeTriplets)]:
    t0 = time.perf_counter(); r = fn(tr, tg)
    print(f"{name:6s}: {r}  {time.perf_counter() - t0:.4f}s")

greedy: True  0.0131s
flags : True  0.0001s
zip   : True  0.0406s
submit: True  0.0071s


## 12. Bộ test

In [16]:
fns = [merge_greedy, merge_flags, merge_zip, lambda a, b: Solution().mergeTriplets(a, b)]

for (tr, tg), want in examples:
    assert merge_subsets(tr, tg) == merge_closure(tr, tg) == want
    for fn in fns:
        assert fn(tr, tg) == want

edge = [
    (([[1, 1, 1]], [1, 1, 1]), True),
    (([[1, 1, 1]], [1, 1, 2]), False),
    (([[1, 2, 3], [3, 2, 1]], [3, 2, 3]), True),
    (([[1, 2, 3], [3, 2, 1]], [2, 2, 2]), False),
    (([[5, 5, 5], [6, 1, 1]], [5, 5, 5]), True),          # bộ độc không cản trở
    (([[1, 1, 9], [9, 1, 1]], [9, 1, 9]), True),
    (([[1, 1, 1000], [1000, 1, 1]], [1000, 1000, 1000]), False),
]
for (tr, tg), want in edge:
    assert merge_subsets(tr, tg) == want, (tr, tg)
    for fn in fns:
        assert fn(tr, tg) == want, (fn, tr, tg)

rng = random.Random(2024)
for _ in range(2000):
    n = rng.randint(1, 7); V = rng.choice([2, 3, 5])
    tr = [[rng.randint(1, V) for _ in range(3)] for _ in range(n)]
    tg = rng.choice(tr + [[rng.randint(1, V) for _ in range(3)]])
    tg = [max(a, b) for a, b in zip(tg, rng.choice(tr))] if rng.random() < 0.5 else tg
    want = merge_subsets(tr, tg)
    for fn in fns:
        assert fn(tr, tg) == want, (fn, tr, tg)
print("ví dụ + edge + 2000 bộ ngẫu nhiên khớp vét cạn tập con")

ví dụ + edge + 2000 bộ ngẫu nhiên khớp vét cạn tập con


In [17]:
# đối chiếu với mô phỏng thao tác thật (BFS trạng thái) trên bộ rất nhỏ
rng = random.Random(7)
for _ in range(150):
    n = rng.randint(1, 3); V = 3
    tr = [[rng.randint(1, V) for _ in range(3)] for _ in range(n)]
    tg = [rng.randint(1, V) for _ in range(3)]
    assert merge_bfs_state(tr, tg) == merge_subsets(tr, tg) == Solution().mergeTriplets(tr, tg)
print("BFS mô phỏng thao tác thật = vét cạn tập con = lời giải (150 bộ)")

BFS mô phỏng thao tác thật = vét cạn tập con = lời giải (150 bộ)

## 13. Lỗi người mới hay gặp

### Lỗi 1 — Không loại bộ độc (lấy max của **mọi** bộ)

In [18]:
def wrong_no_filter(triplets, target):
    return [max(col) for col in zip(*triplets)] == list(target)

for (tr, tg), want in examples:
    print("sai =", wrong_no_filter(tr, tg), "| đúng =", want)

sai = False | đúng = True
sai = False | đúng = False
sai = True | đúng = True


**Sửa:** chỉ lấy max trên các bộ **an toàn**.

### Lỗi 2 — Chỉ kiểm từng thành phần **có xuất hiện** ở đâu đó (không kiểm bộ đó an toàn)

In [19]:
def wrong_any_position(triplets, target):
    return all(any(t[k] == target[k] for t in triplets) for k in range(3))

for (tr, tg), want in examples + [(([[1, 8, 4], [2, 1, 1], [1, 1, 5]], [2, 7, 5]), False)]:
    print("sai =", wrong_any_position(tr, tg), "| đúng =", Solution().mergeTriplets(tr, tg))

sai =

 True | đúng = True
sai = False | đúng = False
sai = True | đúng = True
sai = False | đúng = False


Ở ví dụ 1, giá trị `7` ở vị trí giữa đến từ `[1,7,5]` (an toàn) — tình cờ đúng; nhưng nếu `7` chỉ đến từ một bộ độc thì không dùng được. Hãy tạo phản ví dụ: `[[1,7,9],[2,1,1],[1,1,5]]`, target `[2,7,5]` — `7` chỉ có trong `[1,7,9]` (độc vì `9 > 5`).

In [20]:
tr, tg = [[1, 7, 9], [2, 1, 1], [1, 1, 5]], [2, 7, 5]
print("sai =", wrong_any_position(tr, tg), "| đúng =", Solution().mergeTriplets(tr, tg), "| vét cạn =", merge_subsets(tr, tg))

sai = True | đúng = False | vét cạn = False


**Sửa:** chỉ bật cờ khi bộ đó an toàn.

### Lỗi 3 — Dùng `<` thay vì `≤` khi lọc bộ an toàn

In [21]:
def wrong_strict(triplets, target):
    x, y, z = target; fx = fy = fz = False
    for a, b, c in triplets:
        if a < x and b < y and c < z:                      # SAI: bộ có thành phần BẰNG target bị loại
            fx |= a == x; fy |= b == y; fz |= c == z
    return fx and fy and fz

for (tr, tg), want in examples:
    print("sai =", wrong_strict(tr, tg), "| đúng =", want)

sai = False | đúng = True
sai = False | đúng = False
sai = False | đúng = True


Với `<` nghiêm ngặt, không bộ nào có thể có thành phần **bằng** target ⇒ luôn `False`. **Sửa:** `≤`.

### Lỗi 4 — Tham lam "gộp tuần tự rồi kiểm" (gộp cả bộ độc vào giữa chừng)

In [22]:
def wrong_sequential(triplets, target):
    cur = list(triplets[0])
    for t in triplets[1:]:
        cur = merge(cur, t)                                # SAI: gộp mọi bộ, kể cả bộ độc
        if cur == list(target):
            return True
    return cur == list(target)

for (tr, tg), want in examples:
    print("sai =", wrong_sequential(tr, tg), "| đúng =", want)

sai = False | đúng = True
sai = False | đúng = False
sai = True | đúng = True


**Sửa:** chỉ gộp bộ an toàn (và gộp hết chúng — không cần thứ tự).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Phép gộp **giao hoán + kết hợp + lũy đẳng** (max, min, OR, AND, gcd, lcm) ⇒ kết quả chỉ phụ thuộc **tập** được gộp ⇒ nghĩ theo tập con.
- Phép gộp **đơn điệu** (chỉ tăng) ⇒ phần tử "vượt mục tiêu" là độc, loại ngay.
- "Có tồn tại tập con có max = target" ⇒ lấy max của **mọi** phần tử an toàn.
- Họ bài: 2732 *Find a Good Subset of the Matrix*, 1239, bài OR/AND đích (`bitwise OR = target`: lọc phần tử là "tập con bit" của target).

## 15. Biến thể của bài

1. **Bộ `k` thành phần** thay vì 3: y hệt, dùng mảng cờ độ dài `k`.
2. **Phép gộp là `min`:** an toàn ⇔ mọi thành phần `≥ target`; lấy min của các bộ an toàn.
3. **Phép gộp OR bit:** số `a` an toàn ⇔ `a & ~target == 0`; OR của các số an toàn phải bằng target.
4. **Số bộ ba ít nhất cần dùng** để đạt target: chọn tối thiểu bộ an toàn phủ cả 3 thành phần — tối đa 3 bộ; thử 1, 2, 3 (phủ tập `{x, y, z}` — bài toán phủ nhỏ).

In [23]:
def merge_k(tuples, target):
    """Biến thể 1."""
    k = len(target); found = [False] * k
    for t in tuples:
        if all(a <= b for a, b in zip(t, target)):
            for i in range(k):
                if t[i] == target[i]: found[i] = True
    return all(found)

def or_target(nums, target):
    """Biến thể 3: có tập con OR bằng target không?"""
    acc = 0
    for a in nums:
        if a & ~target == 0:
            acc |= a
    return acc == target

def min_triplets_needed(triplets, target):
    """Biến thể 4: số bộ an toàn ít nhất (1..3) để max = target, hoặc -1."""
    masks = set()
    for t in triplets:
        if all(a <= b for a, b in zip(t, target)):
            masks.add(sum(1 << i for i in range(3) if t[i] == target[i]))
    masks = list(masks)
    for r in (1, 2, 3):
        for comb in combinations(masks, r):
            m = 0
            for x in comb: m |= x
            if m == 7: return r
    return -1

print(merge_k([[1, 2, 3, 4], [4, 2, 1, 1]], [4, 2, 3, 4]), or_target([1, 2, 8], 3), or_target([1, 2, 8], 4))
print("số bộ ít nhất:", [min_triplets_needed(tr, tg) for (tr, tg), _ in examples])

True True False
số bộ ít nhất: [2, -1, 3]


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. BFS trạng thái | mô phỏng thao tác | mũ | mũ | kiểm chứng tí hon |
| 1b. Tập con | max mọi tập con | `O(2ⁿ n)` | `O(n)` | kiểm chứng |
| 1c. Bao đóng | thêm max từng cặp | đa thức theo `U` | `O(U)` | kiểm chứng giá trị nhỏ |
| 2. Tham lam max | lọc + max | `O(n)` | `O(1)` | rõ ràng |
| 3. Ba cờ | đạt đúng từng thành phần | `O(n)` | `O(1)` | **nộp bài** |
| 4. `zip(*)` | lọc + max theo cột | `O(n)` | `O(n)` | gọn |

## 17. Bài tập (từ dễ đến khó)

**Bài 1 (đoán output):** `mergeTriplets([[1,3,4],[2,5,8]], [2,5,8])` và `mergeTriplets([[1,3,1]], [1,3,2])`?

**Bài 2 (tính tay):** với `[[1,5,2],[3,1,4],[3,5,9],[2,2,4]]`, target `[3,5,4]`: phân loại an toàn/độc và kết luận.

**Bài 3 (điền TODO):** hoàn thiện `witness(triplets, target)` trả về **danh sách chỉ số** của một tập con có max đúng bằng target (hoặc `None`).

**Bài 4 (tự viết lại):** viết `merge_min(triplets, target)` cho biến thể phép gộp `min`.

**Bài 5 (mini challenge):** viết `min_ops(triplets, target)` — số **thao tác** ít nhất (mỗi thao tác gộp một cặp) để target xuất hiện. Gợi ý: nếu target đã có thì 0; ngược lại đáp án = (số bộ ít nhất cần dùng) − 1. Kiểm chứng bằng BFS trạng thái mảng trên input tí hon.

In [24]:
def witness(triplets, target):
    # TODO: chỉ số mọi bộ an toàn; kiểm tra max của chúng bằng target
    pass

## 18. Gợi ý 3 mức

**Bài 1:** (1) `[2,5,8]` đã có sẵn ⇒ `True`. (2) `[[1,3,1]]`, target `[1,3,2]`: không bộ nào có `z = 2`. (3) `False`.

**Bài 2:** (1) `[3,5,9]` độc (`9 > 4`). (2) Ba bộ còn lại an toàn: max `[3, 5, 4]`. (3) `True`.

**Bài 3:** (1) `good = [i for i, t in enumerate(triplets) if an toàn]`. (2) Nếu max của `good` bằng target ⇒ trả `good`. (3) Ngược lại `None`.

**Bài 4:** (1) An toàn ⇔ `t[k] ≥ target[k]` mọi `k`. (2) Cờ bật khi `t[k] == target[k]`. (3) Kiểm chứng bằng vét cạn tập con với `min`.

**Bài 5:** (1) Dùng `min_triplets_needed`. (2) Gộp `r` bộ cần `r − 1` thao tác. (3) Target có sẵn ⇒ `0`.

## 19. Đáp án tham khảo

In [25]:
# Bài 1
assert Solution().mergeTriplets([[1, 3, 4], [2, 5, 8]], [2, 5, 8]) is True
assert Solution().mergeTriplets([[1, 3, 1]], [1, 3, 2]) is False

# Bài 2
assert Solution().mergeTriplets([[1, 5, 2], [3, 1, 4], [3, 5, 9], [2, 2, 4]], [3, 5, 4]) is True

# Bài 3
def witness_ans(triplets, target):
    good = [i for i, t in enumerate(triplets) if all(a <= b for a, b in zip(t, target))]
    if good and reduce(merge, [triplets[i] for i in good]) == list(target):
        return good
    return None

# Bài 4
def merge_min(triplets, target):
    found = [False] * 3
    for t in triplets:
        if all(a >= b for a, b in zip(t, target)):
            for k in range(3):
                if t[k] == target[k]: found[k] = True
    return all(found)

def merge_min_brute(triplets, target):
    n = len(triplets)
    return any([min(col) for col in zip(*[triplets[i] for i in comb])] == list(target)
               for r in range(1, n + 1) for comb in combinations(range(n), r))

rng = random.Random(241)
for _ in range(1000):
    tr = [[rng.randint(1, 4) for _ in range(3)] for _ in range(rng.randint(1, 6))]
    tg = [rng.randint(1, 4) for _ in range(3)]
    w = witness_ans(tr, tg)
    assert (w is not None) == Solution().mergeTriplets(tr, tg)
    if w: assert reduce(merge, [tr[i] for i in w]) == tg
    assert merge_min(tr, tg) == merge_min_brute(tr, tg)
print("Bài 1-4: OK")

Bài 1-4: OK


In [26]:
# Bài 5 — số thao tác ít nhất
def min_ops(triplets, target):
    if list(target) in [list(t) for t in triplets]:
        return 0
    r = min_triplets_needed(triplets, target)
    return -1 if r == -1 else r - 1

def min_ops_bfs(triplets, target):
    start = tuple(tuple(t) for t in triplets); tg = tuple(target)
    dist = {start: 0}; q = deque([start])
    while q:
        st = q.popleft()
        if tg in st: return dist[st]
        for i in range(len(st)):
            for j in range(len(st)):
                if i != j:
                    nt = tuple(max(a, b) for a, b in zip(st[i], st[j]))
                    ns = st[:j] + (nt,) + st[j + 1:]
                    if ns not in dist:
                        dist[ns] = dist[st] + 1; q.append(ns)
    return -1

rng = random.Random(242)
for _ in range(200):
    tr = [[rng.randint(1, 3) for _ in range(3)] for _ in range(rng.randint(1, 4))]
    tg = [rng.randint(1, 3) for _ in range(3)]
    assert min_ops(tr, tg) == min_ops_bfs(tr, tg), (tr, tg)
print("Bài 5: khớp BFS; ví dụ 3 cần", min_ops(*examples[2][0]), "thao tác")

Bài 5: khớp BFS; ví dụ 3 cần 2 thao tác


## 20. Template nhớ nhanh + Checklist

```python
x, y, z = target
fx = fy = fz = False
for a, b, c in triplets:
    if a <= x and b <= y and c <= z:       # chỉ bộ AN TOÀN (không vượt target)
        fx |= a == x; fy |= b == y; fz |= c == z
return fx and fy and fz
```

**Checklist trước khi submit:**
- [ ] Loại bộ có **bất kỳ** thành phần vượt target.
- [ ] So sánh `≤` (không phải `<`) khi lọc.
- [ ] Mỗi thành phần target phải được đạt **đúng** bởi một bộ an toàn.
- [ ] Không gộp tuần tự mọi bộ (bộ độc phá hỏng).
- [ ] Target có sẵn trong mảng ⇒ `True`.